# Silver layer

The silver layer is build up like this:  
1.  import bronze df and check.  
2.  quick sanity checks, col present, missing val, data type.  
3.  Define doc boundary and silver schema.  
4.  one coherent text (silver_text).  
5.  inspect silver_text.  
6.  what cleaning is allowed?  
7.  start preprocessing.  
8.  spaCY annotation.  
9.  output to gold.  

In [105]:
# Import needed libraries
import pandas as pd
import spacy
import os
import re
import unicodedata
from bs4 import BeautifulSoup

# Import bronze dataframe + sanity checks

In [106]:
# Load bronze data
project_data = pd.read_csv("../Data/bronze/bronze_data.csv")

# check the required columns
required_columns = ['id', 'content', 'title']
missing_columns = list(set(required_columns) - set(project_data.columns))

if missing_columns:
    print(f"Error: Missing columns in the dataset: {missing_columns}")
else:
    print(f"All required columns are present: {list(project_data.columns)}")

# Print project_data info
print(project_data.info())

# Check for missing values in the required columns
missing_values = project_data[required_columns].isnull().sum()
print(f"Missing values in required columns:\n{missing_values}")



All required columns are present: ['id', 'title', 'content']
<class 'pandas.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   id       1 non-null      str  
 1   title    1 non-null      str  
 2   content  1 non-null      str  
dtypes: str(3)
memory usage: 156.0 bytes
None
Missing values in required columns:
id         0
content    0
title      0
dtype: int64


# Clean incoming text data

first we will check and fill the na's

In [107]:
project_data["id"] = project_data["id"].fillna("").astype(str)
project_data["title"] = project_data["title"].fillna("").astype(str)
project_data["content"] = project_data["content"].fillna("").astype(str)

then we clean the id column

In [108]:
# clean the id column by stripping leading and trailing whitespace
project_data["id"] = project_data["id"].str.strip()

then we decapitalize the titles so its easier to process

In [109]:
# we will decapitalize the title to make it easier for the model to process the text.
# we only do the title because lowercasing content can hurt the NER performance as it can remove important capitalization cues.
project_data['title'] = project_data['title'].str.lower()
print(project_data['title'].head())

0    assessing  climate adaptation measures in low-...
Name: title, dtype: str


furthermore, we normalize the unicodes

In [110]:
# fix the encoding unicode normalization issues in the content column
def normalize_unicode(text: str) -> str:
    if not text:
        return ""
    text = unicodedata.normalize('NFKC', text)
    text = (text.replace("\u00A0", " ")   # NBSP
                .replace("“", '"').replace("”", '"')
                .replace("‘", "'").replace("’", "'")
                .replace("—", "-").replace("–", "-"))
    # remove control chars except newline/tab
    text = re.sub(r"[\x00-\x08\x0B\x0C\x0E-\x1F]", "", text)
    return text

project_data['content'] = project_data['content'].apply(normalize_unicode)
project_data['title'] = project_data['title'].apply(normalize_unicode)

Then we strip any existing html codes within the data

In [111]:
def strip_html(text: str) -> str:
    if not text:
        return ""

    soup = BeautifulSoup(text, "html.parser")

    # Replace <br> with newline before removing tags
    for br in soup.find_all("br"):
        br.replace_with("\n")

    cleaned = soup.get_text()
    return cleaned

project_data["title"] = project_data["title"].apply(strip_html)
project_data["content"] = project_data["content"].apply(strip_html)

Then we normalize any whitespace

In [112]:
def normalize_whitespace(text: str) -> str:
    if not text:
        return ""
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)        # collapse spaces/tabs
    text = re.sub(r"\n{3,}", "\n\n", text)     # keep paragraph breaks
    return text.strip()

project_data["title"] = project_data["title"].apply(normalize_whitespace)
project_data["content"] = project_data["content"].apply(normalize_whitespace)

follwing that up with normalizing punctuation

In [113]:
def normalize_punctuation(text: str) -> str:
    if not text:
        return ""

    # fix glued sentence boundary: "methods.Policy" -> "methods. Policy"
    text = re.sub(r"([a-z])\.([A-Z])", r"\1. \2", text)

    # collapse repeated punctuation
    text = re.sub(r"\.{2,}", ".", text)
    text = re.sub(r"!{2,}", "!", text)
    text = re.sub(r"\?{2,}", "?", text)

    # normalize spacing around punctuation
    text = re.sub(r"\s+([,.;:!?])", r"\1", text)              # no space before punct
    text = re.sub(r"([,.;:!?])([A-Za-z])", r"\1 \2", text)    # ensure space after punct

    return text

project_data["content"] = project_data["content"].apply(normalize_punctuation)
project_data["title"] = project_data["title"].apply(normalize_punctuation)

In [114]:
def smart_title_case(s: str) -> str:
    small = {"and", "or", "the", "a", "an", "of", "in", "on", "to", "for", "with"}
    words = s.split()
    out = []
    for i, w in enumerate(words):
        lw = w.lower()
        if i > 0 and lw in small:
            out.append(lw)
        else:
            out.append(w[:1].upper() + w[1:].lower())
    return " ".join(out)

def normalize_headings(text: str) -> str:
    if not text:
        return ""
    lines = text.split("\n")
    out = []
    for line in lines:
        raw = line.strip()

        # heuristic: headings are short lines mostly letters/spaces/&/-
        if raw and len(raw) <= 70 and re.fullmatch(r"[A-Za-z0-9 &\-\(\)]+", raw):
            # if it looks like a heading (not a full sentence)
            if raw.count(" ") <= 8 and not raw.endswith("."):
                # normalize & to "and" if you want consistency
                raw2 = raw.replace("&", "and")
                out.append(smart_title_case(raw2))
                continue
        out.append(line)
    return "\n".join(out)

project_data["content"] = project_data["content"].apply(normalize_headings)

In [115]:
project_data["content"] = project_data["content"].apply(normalize_whitespace)

Now we imported the data correctly, we go on to the pre-NLP preprocessing steps.  
We make the data ready for NLP usage, this mean we will check for:  
- concatenate to text.    
- Normalize the whitespace.  
- Split paragraphs.  
- dedupe prarapraphs.  

In [116]:
# First we will concatenate the title and content columns to create a single text column for processing
def concat_to_text(x):
    if x is None:
        return ""
    # x is expected to be [title, content]
    parts = [str(part).strip() for part in x if part is not None and str(part).strip()]
    return "\n\n".join(parts)

# Concatenate title and content into a new text column
project_data['text'] = project_data.apply(lambda row: concat_to_text([row['title'], row['content']]), axis=1)
print(project_data['text'])

print(project_data.head())

0    assessing climate adaptation measures in low-l...
Name: text, dtype: str
       id                                              title  \
0  DOC_01  assessing climate adaptation measures in low-l...   

                                             content  \
0  Introduction\nClimate adaptation has become a ...   

                                                text  
0  assessing climate adaptation measures in low-l...  


Then we dedupe any paragraphs that have been writting twice by accident

In [117]:
def dedupe_paragraphs(text: str) -> str:
    if not text:
        return ""
    paras = [p.strip() for p in text.split("\n\n") if p.strip()]
    seen = set()
    out = []
    for p in paras:
        key = re.sub(r"\s+", " ", p).strip().lower()
        if key not in seen:
            seen.add(key)
            out.append(p)
    return "\n\n".join(out)

project_data["text"] = project_data["text"].apply(dedupe_paragraphs)

In [118]:
# now we can create the silver_text column which will be used for the silver layer
project_data['silver_text'] = project_data['text']
print(project_data['silver_text'])

print(project_data.head())


0    assessing climate adaptation measures in low-l...
Name: silver_text, dtype: str
       id                                              title  \
0  DOC_01  assessing climate adaptation measures in low-l...   

                                             content  \
0  Introduction\nClimate adaptation has become a ...   

                                                text  \
0  assessing climate adaptation measures in low-l...   

                                         silver_text  
0  assessing climate adaptation measures in low-l...  


In [119]:
# check length distribution
project_data["silver_len"] = project_data["silver_text"].str.len()
print(project_data["silver_len"].describe())

# show a sample
print(project_data.loc[0, "silver_text"][:1500])

count       1.0
mean     5985.0
std         NaN
min      5985.0
25%      5985.0
50%      5985.0
75%      5985.0
max      5985.0
Name: silver_len, dtype: float64
assessing climate adaptation measures in low-lying coastal regions

Introduction
Climate adaptation has become a central topic in regional planning as coastal areas face increasing risks from sea level rise, extreme precipitation, and prolonged droughts. Low lying coastal regions are particulary vulnerable due to their exposure to flooding, saltwater intrusion, and land subsidence. These risks threaten not only physical infrastructure but also ecosystems, economic activities and the livability of coastal communities. As climate impacts intensify regional authorities are under growing pressure to implement effective adaptation measures.

Background and Context
Over the past decade, a wide range of climate adaptation strategies have been proposed & implemented in coastal regions. These strategies include hard infrastructure solut

# sanity checks

In [120]:
html_left = project_data["silver_text"].str.contains(r"<[^>]+>").sum()
print("HTML tags remaining in content:", html_left)

HTML tags remaining in content: 0


In [121]:
# Select only Silver-relevant columns
silver_df = project_data[["id", "silver_text"]].copy()

# Reset index for clean export
silver_df = silver_df.reset_index(drop=True)

print(silver_df.head())

       id                                        silver_text
0  DOC_01  assessing climate adaptation measures in low-l...


In [122]:
silver_path = "../Data/silver/silver_data.csv"

silver_df.to_csv(silver_path, index=False, encoding="utf-8")

print(f"Silver data saved to: {silver_path}")

Silver data saved to: ../Data/silver/silver_data.csv


# TO-DO

- Add data preprocessing pipeline. (duplication etc...) """""CHECK"""""
- Add NLP processing.  